# STAC + WRS Path/Row Filter Demo (Planetary Computer)

This notebook demonstrates a **working pattern** for Landsat item selection when you need
strict `landsat:wrs_path` and `landsat:wrs_row` constraints:

1. Query broad candidates from STAC
2. Filter by month + WRS path/row on item properties (client-side)
3. Optionally test server-side WRS query support
4. Load one filtered scene with `odc.stac.load` as proof


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pystac_client
import planetary_computer
import rioxarray
import odc.stac

pd.set_option('display.max_rows', 200)
plt.rcParams['figure.dpi'] = 120
print('✅ Imports ready')


In [ ]:
# -------------------------------
# CONFIG
# -------------------------------
YEAR = 2005
DATE_RANGE = '2004-01-01/2006-12-31'  # Paper-aligned window for 2005 map support
CLOUD_MAX = 25.0
ALLOWED_MONTHS = {1, 2, 3, 4, 5, 6, 9, 10, 11, 12}  # compfile-style seasonal coverage

# WRS pairs from supplementary-style scene planning (edit as needed for your AOI)
ALLOWED_WRS = {
    (145, 38), (145, 39),
    (146, 38), (146, 39),
    (147, 38), (147, 39),
    (148, 39),
}

MASK_PATH = Path('/Users/neerajkaroshi/Desktop/Projects/clay_LULC/data/processed/lulc_masks/uk_2005_30m.tif')

if not MASK_PATH.exists():
    raise FileNotFoundError(f'Mask not found: {MASK_PATH}')

mask = rioxarray.open_rasterio(MASK_PATH)
bbox = mask.rio.transform_bounds('EPSG:4326')
print('📌 BBOX (WGS84):', bbox)
print('📌 Date range   :', DATE_RANGE)
print('📌 Allowed WRS  :', sorted(list(ALLOWED_WRS))[:8], '...')


In [ ]:
def item_cloud(it):
    return float(it.properties.get('eo:cloud_cover', 100.0) or 100.0)


def item_month(it):
    dt = it.datetime
    return dt.month if dt is not None else None


def item_wrs(it):
    p = it.properties.get('landsat:wrs_path', None)
    r = it.properties.get('landsat:wrs_row', None)
    if p is None or r is None:
        return None
    return int(p), int(r)


def item_sensor(it):
    return it.properties.get('platform', 'unknown')


def item_date(it):
    return it.datetime.isoformat() if it.datetime is not None else 'na'


In [ ]:
# -------------------------------
# STEP 1: Broad STAC search
# -------------------------------
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=['landsat-c2-l2'],
    bbox=bbox,
    datetime=DATE_RANGE,
    query={'eo:cloud_cover': {'lte': CLOUD_MAX}},
    max_items=5000,
)
items_raw = list(search.items())
print(f'🔎 Raw STAC items: {len(items_raw)}')


In [ ]:
# -------------------------------
# STEP 2: Client-side filter by month + WRS path/row
# -------------------------------
items_filt = [
    it for it in items_raw
    if item_month(it) in ALLOWED_MONTHS
    and item_wrs(it) in ALLOWED_WRS
]

print(f'✅ Filtered items (month + WRS): {len(items_filt)}')

rows = []
for it in items_filt:
    wrs = item_wrs(it)
    rows.append({
        'id': it.id,
        'datetime': item_date(it),
        'cloud': item_cloud(it),
        'platform': item_sensor(it),
        'wrs_path': wrs[0] if wrs else None,
        'wrs_row': wrs[1] if wrs else None,
    })

if rows:
    df = pd.DataFrame(rows).sort_values(['cloud', 'datetime']).reset_index(drop=True)
    print('Top candidates after filter:')
    display(df.head(20))
else:
    print('⚠️ No items after filter. Expand WRS/month/date bounds and rerun.')


In [ ]:
# -------------------------------
# STEP 3 (optional): server-side WRS query support test
# -------------------------------
# Some STAC backends support this query directly; some do not.
# Keep this as diagnostic only.

def try_server_side_wrs_query(sample_paths, sample_rows):
    try:
        s2 = catalog.search(
            collections=['landsat-c2-l2'],
            bbox=bbox,
            datetime=DATE_RANGE,
            query={
                'eo:cloud_cover': {'lte': CLOUD_MAX},
                'landsat:wrs_path': {'in': sample_paths},
                'landsat:wrs_row': {'in': sample_rows},
            },
            max_items=2000,
        )
        it2 = list(s2.items())
        print(f'ℹ️ Server-side WRS query returned: {len(it2)} items')
        return it2
    except Exception as e:
        print('ℹ️ Server-side WRS query not supported here:', e)
        return []

_ = try_server_side_wrs_query(
    sample_paths=sorted({p for p, _ in ALLOWED_WRS}),
    sample_rows=sorted({r for _, r in ALLOWED_WRS}),
)


In [ ]:
# -------------------------------
# STEP 4: Proof load with odc.stac.load
# -------------------------------
if not items_filt:
    raise RuntimeError('No filtered items available; cannot run load demo.')

# choose clearest item
best = sorted(items_filt, key=lambda it: item_cloud(it))[0]
print('🧪 Testing scene:', best.id)
print('   datetime:', item_date(best))
print('   cloud   :', item_cloud(best))
print('   wrs     :', item_wrs(best))

# Small central tile from mask for quick proof load
ny, nx = mask.sizes['y'], mask.sizes['x']
i0 = max(0, (ny // 2) - 128)
j0 = max(0, (nx // 2) - 128)
lbl_tile = mask.isel(y=slice(i0, i0 + 256), x=slice(j0, j0 + 256))

signed_item = planetary_computer.sign(best)
ds = odc.stac.load(
    [signed_item],
    geobox=lbl_tile.odc.geobox,
    bands=['red', 'green', 'blue', 'nir08', 'swir16', 'swir22'],
    resampling='bilinear',
    fail_on_error=False,
).squeeze().compute()

arr = ds.to_array().values  # [6, 256, 256]
print('✅ odc.stac.load success | shape:', arr.shape)

# quick RGB preview
rgb = np.stack([arr[0], arr[1], arr[2]], axis=-1).astype(np.float32)
for k in range(3):
    b = rgb[..., k]
    p2, p98 = np.nanpercentile(b, [2, 98])
    rgb[..., k] = np.clip((b - p2) / (p98 - p2 + 1e-6), 0, 1)

plt.figure(figsize=(6, 6))
plt.imshow(np.nan_to_num(rgb, nan=0.0))
plt.title(f'RGB Proof Tile | {best.id}')
plt.axis('off')
plt.show()


## Conclusion

If `items_filt` is non-empty and `odc.stac.load` succeeds, your pipeline can safely use:

- broad STAC search
- strict client-side WRS + season filters
- then load only approved items

This is the recommended approach when exact `path/row` filtering is required for paper-faithful data selection.
